In [1]:
import os
import numpy as np
import pandas as pd

RANDOM_STATE = 42

DATA_DIR_CANDIDATES = [
    "/kaggle/data/champs-scalar-coupling",
    "/kaggle/input/champs-scalar-coupling",
    "/kaggle/data",
    "/kaggle/input",
]


def find_file(filename: str) -> str:
    for d in DATA_DIR_CANDIDATES:
        p = os.path.join(d, filename)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        f"Could not find {filename} in any of: {DATA_DIR_CANDIDATES}"
    )


train_path = find_file("train.csv")
test_path = find_file("test.csv")
structures_path = find_file("structures.csv")
sample_sub_path = find_file("sample_submission.csv")

mulliken_path = find_file("mulliken_charges.csv")
magshield_path = find_file("magnetic_shielding_tensors.csv")
dipole_path = find_file("dipole_moments.csv")
potential_path = find_file("potential_energy.csv")

scc_contrib_path = find_file("scalar_coupling_contributions.csv")

print("Using paths:")
print("train:", train_path)
print("test:", test_path)
print("structures:", structures_path)
print("sample_submission:", sample_sub_path)
print("mulliken:", mulliken_path)
print("magnetic_shielding_tensors:", magshield_path)
print("dipole_moments:", dipole_path)
print("potential_energy:", potential_path)
print("scalar_coupling_contributions:", scc_contrib_path)



Using paths:
train: /kaggle/data/champs-scalar-coupling/train.csv
test: /kaggle/data/champs-scalar-coupling/test.csv
structures: /kaggle/data/champs-scalar-coupling/structures.csv
sample_submission: /kaggle/data/champs-scalar-coupling/sample_submission.csv
mulliken: /kaggle/data/champs-scalar-coupling/mulliken_charges.csv
magnetic_shielding_tensors: /kaggle/data/champs-scalar-coupling/magnetic_shielding_tensors.csv
dipole_moments: /kaggle/data/champs-scalar-coupling/dipole_moments.csv
potential_energy: /kaggle/data/champs-scalar-coupling/potential_energy.csv
scalar_coupling_contributions: /kaggle/data/champs-scalar-coupling/scalar_coupling_contributions.csv


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

# Speed: parse categorical cols as pandas 'category' (faster + smaller than string/object),
# while preserving exact categorical values/semantics for OneHotEncoder.
train = pd.read_csv(
    train_path,
    usecols=[
        "id",
        "molecule_name",
        "atom_index_0",
        "atom_index_1",
        "type",
        "scalar_coupling_constant",
    ],
    dtype={
        "id": np.int32,
        "molecule_name": "category",
        "atom_index_0": np.int16,
        "atom_index_1": np.int16,
        "type": "category",
    },
)
test = pd.read_csv(
    test_path,
    usecols=["id", "molecule_name", "atom_index_0", "atom_index_1", "type"],
    dtype={
        "id": np.int32,
        "molecule_name": "category",
        "atom_index_0": np.int16,
        "atom_index_1": np.int16,
        "type": "category",
    },
)
structures = pd.read_csv(
    structures_path,
    usecols=["molecule_name", "atom_index", "atom", "x", "y", "z"],
    dtype={
        "molecule_name": "category",
        "atom_index": np.int16,
        "atom": "category",
        "x": np.float64,
        "y": np.float64,
        "z": np.float64,
    },
)

mulliken = pd.read_csv(
    mulliken_path,
    usecols=["molecule_name", "atom_index", "mulliken_charge"],
    dtype={
        "molecule_name": "category",
        "atom_index": np.int16,
        "mulliken_charge": np.float64,
    },
)
mag_cols = ["XX", "YX", "ZX", "XY", "YY", "ZY", "XZ", "YZ", "ZZ"]
magshield = pd.read_csv(
    magshield_path,
    usecols=["molecule_name", "atom_index"] + mag_cols,
    dtype={
        "molecule_name": "category",
        "atom_index": np.int16,
        **{c: np.float64 for c in mag_cols},
    },
)
dipole = pd.read_csv(
    dipole_path,
    usecols=["molecule_name", "X", "Y", "Z"],
    dtype={
        "molecule_name": "category",
        "X": np.float64,
        "Y": np.float64,
        "Z": np.float64,
    },
)
potential = pd.read_csv(
    potential_path,
    usecols=["molecule_name", "potential_energy"],
    dtype={"molecule_name": "category", "potential_energy": np.float64},
)

scc = pd.read_csv(
    scc_contrib_path,
    usecols=[
        "molecule_name",
        "atom_index_0",
        "atom_index_1",
        "type",
        "fc",
        "sd",
        "pso",
        "dso",
    ],
    dtype={
        "molecule_name": "category",
        "atom_index_0": np.int16,
        "atom_index_1": np.int16,
        "type": "category",
        "fc": np.float64,
        "sd": np.float64,
        "pso": np.float64,
        "dso": np.float64,
    },
)

print(train.shape, test.shape, structures.shape)
print(
    "aux shapes:",
    mulliken.shape,
    magshield.shape,
    dipole.shape,
    potential.shape,
    scc.shape,
)



(4191263, 6) (467813, 5) (1379964, 6)
aux shapes: (1379964, 3) (1379964, 11) (76510, 4) (76510, 2) (4191263, 8)


In [3]:
mol_centroid = (
    structures.groupby("molecule_name", sort=False)[["x", "y", "z"]]
    .mean()
    .rename(columns={"x": "cx", "y": "cy", "z": "cz"})
)

ATOM_Z = {
    "H": 1,
    "C": 6,
    "N": 7,
    "O": 8,
    "F": 9,
    "P": 15,
    "S": 16,
    "Cl": 17,
    "Br": 35,
    "I": 53,
}


# Speed: replace per-molecule NxN distance matrix with an exact grid-based neighbor search.
# This preserves correctness because it still computes *exact* Euclidean distances and applies
# the same (dist <= cutoff) rule; it just avoids evaluating pairs that cannot be within cutoff.
def build_atom_neighbor_features_both(
    structures_df: pd.DataFrame, cutoff1: float = 2.0, cutoff2: float = 4.0
) -> pd.DataFrame:
    coords = structures_df[
        ["molecule_name", "atom_index", "atom", "x", "y", "z"]
    ].copy()
    coords["atom"] = coords["atom"].astype(str)  # stable mapping vs category codes

    suf1 = f"_{cutoff1:.1f}".replace(".", "p")
    suf2 = f"_{cutoff2:.1f}".replace(".", "p")

    max_cut = float(max(cutoff1, cutoff2))
    cell = max_cut  # safe: only need to check adjacent cells in [-1,0,1]^3

    out_blocks = []

    for mol, g in coords.groupby("molecule_name", sort=False):
        g = g.sort_values("atom_index", kind="mergesort")
        idx = g["atom_index"].to_numpy()
        xyz = g[["x", "y", "z"]].to_numpy(dtype=np.float64, copy=False)
        atoms = g["atom"].to_numpy()

        n = len(g)
        if n == 0:
            continue

        # Precompute atom-type boolean masks once per molecule
        is_H = atoms == "H"
        is_C = atoms == "C"
        is_N = atoms == "N"
        is_O = atoms == "O"
        is_other = ~(is_H | is_C | is_N | is_O)

        # Grid indexing
        grid = np.floor(xyz / cell).astype(np.int32, copy=False)
        keys = (
            (grid[:, 0].astype(np.int64) << 42)
            ^ (grid[:, 1].astype(np.int64) << 21)
            ^ grid[:, 2].astype(np.int64)
        )
        order = np.argsort(keys, kind="mergesort")
        keys_sorted = keys[order]

        # Build cell -> slice mapping in sorted order
        unique_keys, start_idx = np.unique(keys_sorted, return_index=True)
        end_idx = np.empty_like(start_idx)
        end_idx[:-1] = start_idx[1:]
        end_idx[-1] = n

        # dict from key to (start,end) indices into 'order' (small per molecule)
        cell_slices = {
            int(k): (int(s), int(e)) for k, s, e in zip(unique_keys, start_idx, end_idx)
        }

        # Accumulators for both cutoffs
        deg1 = np.zeros(n, dtype=np.int32)
        deg2 = np.zeros(n, dtype=np.int32)

        sumd1 = np.zeros(n, dtype=np.float64)
        sumd2 = np.zeros(n, dtype=np.float64)

        mind1 = np.full(n, np.inf, dtype=np.float64)
        mind2 = np.full(n, np.inf, dtype=np.float64)

        # Neighbor type counts
        nH1 = np.zeros(n, dtype=np.int16)
        nC1 = np.zeros(n, dtype=np.int16)
        nN1 = np.zeros(n, dtype=np.int16)
        nO1 = np.zeros(n, dtype=np.int16)
        nX1 = np.zeros(n, dtype=np.int16)

        nH2 = np.zeros(n, dtype=np.int16)
        nC2 = np.zeros(n, dtype=np.int16)
        nN2 = np.zeros(n, dtype=np.int16)
        nO2 = np.zeros(n, dtype=np.int16)
        nX2 = np.zeros(n, dtype=np.int16)

        # Iterate cells; for each atom, compare only with atoms in its 27 neighboring cells
        # Use squared distances to filter, and only sqrt for candidates.
        c1_2 = cutoff1 * cutoff1
        c2_2 = cutoff2 * cutoff2

        # Precompute neighbor cell key deltas
        deltas = [
            (dx, dy, dz) for dx in (-1, 0, 1) for dy in (-1, 0, 1) for dz in (-1, 0, 1)
        ]

        # For each atom, fetch candidate indices from neighbor cells
        for i in range(n):
            gx, gy, gz = grid[i]
            cand = []
            for dx, dy, dz in deltas:
                k = (
                    (np.int64(gx + dx) << 42)
                    ^ (np.int64(gy + dy) << 21)
                    ^ np.int64(gz + dz)
                )
                sl = cell_slices.get(int(k))
                if sl is not None:
                    s, e = sl
                    cand.append(order[s:e])
            if not cand:
                continue
            cand_idx = np.concatenate(cand)
            # remove self if present
            cand_idx = cand_idx[cand_idx != i]
            if cand_idx.size == 0:
                continue

            dif = xyz[cand_idx] - xyz[i]
            d2 = np.einsum("ij,ij->i", dif, dif)
            # For cutoff2: candidates within 4.0
            mask2 = d2 <= c2_2
            if mask2.any():
                j2 = cand_idx[mask2]
                d2_ = d2[mask2]
                d_ = np.sqrt(d2_, dtype=np.float64)

                deg2[i] = deg2[i] + j2.size
                sumd2[i] = sumd2[i] + d_.sum()
                md = d_.min()
                if md < mind2[i]:
                    mind2[i] = md

                # Type counts (cutoff2)
                nH2[i] = nH2[i] + int(is_H[j2].sum())
                nC2[i] = nC2[i] + int(is_C[j2].sum())
                nN2[i] = nN2[i] + int(is_N[j2].sum())
                nO2[i] = nO2[i] + int(is_O[j2].sum())
                nX2[i] = nX2[i] + int(is_other[j2].sum())

                # For cutoff1: subset of those also within 2.0
                mask1 = d2_ <= c1_2
                if mask1.any():
                    j1 = j2[mask1]
                    d1_ = d_[mask1]
                    deg1[i] = deg1[i] + j1.size
                    sumd1[i] = sumd1[i] + d1_.sum()
                    md1 = d1_.min()
                    if md1 < mind1[i]:
                        mind1[i] = md1

                    nH1[i] = nH1[i] + int(is_H[j1].sum())
                    nC1[i] = nC1[i] + int(is_C[j1].sum())
                    nN1[i] = nN1[i] + int(is_N[j1].sum())
                    nO1[i] = nO1[i] + int(is_O[j1].sum())
                    nX1[i] = nX1[i] + int(is_other[j1].sum())

        # Convert sums/mins to mean/min with NaNs when degree=0 (to match original nanmean/nanmin behavior)
        mean1 = np.full(n, np.nan, dtype=np.float64)
        mean2 = np.full(n, np.nan, dtype=np.float64)

        non0_1 = deg1 > 0
        non0_2 = deg2 > 0
        mean1[non0_1] = sumd1[non0_1] / deg1[non0_1]
        mean2[non0_2] = sumd2[non0_2] / deg2[non0_2]

        min1 = np.where(non0_1, mind1, np.nan)
        min2 = np.where(non0_2, mind2, np.nan)

        block = pd.DataFrame(
            {
                "molecule_name": mol,
                "atom_index": idx,
                f"degree{suf1}": deg1,
                f"neigh_dist_mean{suf1}": mean1,
                f"neigh_dist_min{suf1}": min1,
                f"neigh_H{suf1}": nH1,
                f"neigh_C{suf1}": nC1,
                f"neigh_N{suf1}": nN1,
                f"neigh_O{suf1}": nO1,
                f"neigh_other{suf1}": nX1,
                f"degree{suf2}": deg2,
                f"neigh_dist_mean{suf2}": mean2,
                f"neigh_dist_min{suf2}": min2,
                f"neigh_H{suf2}": nH2,
                f"neigh_C{suf2}": nC2,
                f"neigh_N{suf2}": nN2,
                f"neigh_O{suf2}": nO2,
                f"neigh_other{suf2}": nX2,
            }
        )
        out_blocks.append(block)

    if not out_blocks:
        cols = ["molecule_name", "atom_index"]
        for suf in (suf1, suf2):
            cols += [
                f"degree{suf}",
                f"neigh_dist_mean{suf}",
                f"neigh_dist_min{suf}",
                f"neigh_H{suf}",
                f"neigh_C{suf}",
                f"neigh_N{suf}",
                f"neigh_O{suf}",
                f"neigh_other{suf}",
            ]
        return pd.DataFrame(columns=cols)

    return pd.concat(out_blocks, axis=0, ignore_index=True)


atom_env = build_atom_neighbor_features_both(structures, cutoff1=2.0, cutoff2=4.0)
print("Atom env features (both cutoffs):", atom_env.shape)



/tmp/ipykernel_11/295827160.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  structures.groupby("molecule_name", sort=False)[["x", "y", "z"]]
/tmp/ipykernel_11/295827160.py:40: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for mol, g in coords.groupby("molecule_name", sort=False):


Atom env features (both cutoffs): (1379964, 18)


In [4]:
mulliken_feat = mulliken.copy()

magshield_feat = magshield.copy()
magshield_feat["ms_trace"] = (
    magshield_feat["XX"] + magshield_feat["YY"] + magshield_feat["ZZ"]
)
magshield_feat["ms_frob"] = np.sqrt(
    (magshield_feat[mag_cols].to_numpy(dtype=np.float64, copy=False) ** 2).sum(axis=1)
)

dipole_feat = dipole.copy()
dipole_feat["dipole_norm"] = np.sqrt(
    (dipole_feat[["X", "Y", "Z"]].to_numpy(dtype=np.float64, copy=False) ** 2).sum(
        axis=1
    )
)

potential_feat = potential.copy()

print(
    "Prepared aux feats:",
    mulliken_feat.shape,
    magshield_feat.shape,
    dipole_feat.shape,
    potential_feat.shape,
)



Prepared aux feats: (1379964, 3) (1379964, 13) (76510, 5) (76510, 2)


In [5]:
s0 = structures.rename(
    columns={
        "atom_index": "atom_index_0",
        "atom": "atom_0",
        "x": "x0",
        "y": "y0",
        "z": "z0",
    }
)[["molecule_name", "atom_index_0", "atom_0", "x0", "y0", "z0"]]

s1 = structures.rename(
    columns={
        "atom_index": "atom_index_1",
        "atom": "atom_1",
        "x": "x1",
        "y": "y1",
        "z": "z1",
    }
)[["molecule_name", "atom_index_1", "atom_1", "x1", "y1", "z1"]]

env0 = atom_env.rename(columns={"atom_index": "atom_index_0"}).copy()
env1 = atom_env.rename(columns={"atom_index": "atom_index_1"}).copy()
env1 = env1.rename(
    columns={
        c: (c + "_1")
        for c in env1.columns
        if c not in ["molecule_name", "atom_index_1"]
    }
)

m0 = mulliken_feat.rename(
    columns={"atom_index": "atom_index_0", "mulliken_charge": "mulliken_0"}
)[["molecule_name", "atom_index_0", "mulliken_0"]]
m1 = mulliken_feat.rename(
    columns={"atom_index": "atom_index_1", "mulliken_charge": "mulliken_1"}
)[["molecule_name", "atom_index_1", "mulliken_1"]]

ms0 = magshield_feat.rename(columns={"atom_index": "atom_index_0"})[
    ["molecule_name", "atom_index_0"] + mag_cols + ["ms_trace", "ms_frob"]
]
ms1 = magshield_feat.rename(columns={"atom_index": "atom_index_1"})[
    ["molecule_name", "atom_index_1"] + mag_cols + ["ms_trace", "ms_frob"]
].copy()
ms1 = ms1.rename(
    columns={
        c: (c + "_1") for c in ms1.columns if c not in ["molecule_name", "atom_index_1"]
    }
)

dipole_m = dipole_feat.rename(
    columns={"X": "dipole_X", "Y": "dipole_Y", "Z": "dipole_Z"}
)[["molecule_name", "dipole_X", "dipole_Y", "dipole_Z", "dipole_norm"]]
potential_m = potential_feat[["molecule_name", "potential_energy"]].copy()

assert not s0.duplicated(["molecule_name", "atom_index_0"]).any()
assert not s1.duplicated(["molecule_name", "atom_index_1"]).any()
assert not env0.duplicated(["molecule_name", "atom_index_0"]).any()
assert not env1.duplicated(["molecule_name", "atom_index_1"]).any()
assert not m0.duplicated(["molecule_name", "atom_index_0"]).any()
assert not m1.duplicated(["molecule_name", "atom_index_1"]).any()
assert not ms0.duplicated(["molecule_name", "atom_index_0"]).any()
assert not ms1.duplicated(["molecule_name", "atom_index_1"]).any()


def add_pair_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out = out.merge(s0, on=["molecule_name", "atom_index_0"], how="left", sort=False)
    out = out.merge(s1, on=["molecule_name", "atom_index_1"], how="left", sort=False)

    out = out.merge(
        mol_centroid.reset_index(), on="molecule_name", how="left", sort=False
    )

    out = out.merge(env0, on=["molecule_name", "atom_index_0"], how="left", sort=False)
    out = out.merge(env1, on=["molecule_name", "atom_index_1"], how="left", sort=False)

    out = out.merge(m0, on=["molecule_name", "atom_index_0"], how="left", sort=False)
    out = out.merge(m1, on=["molecule_name", "atom_index_1"], how="left", sort=False)

    out = out.merge(ms0, on=["molecule_name", "atom_index_0"], how="left", sort=False)
    out = out.merge(ms1, on=["molecule_name", "atom_index_1"], how="left", sort=False)

    out = out.merge(dipole_m, on="molecule_name", how="left", sort=False)
    out = out.merge(potential_m, on="molecule_name", how="left", sort=False)

    dx = out["x0"] - out["x1"]
    dy = out["y0"] - out["y1"]
    dz = out["z0"] - out["z1"]

    out["dx"] = dx
    out["dy"] = dy
    out["dz"] = dz

    out["abs_dx"] = dx.abs()
    out["abs_dy"] = dy.abs()
    out["abs_dz"] = dz.abs()

    out["dist2"] = dx * dx + dy * dy + dz * dz
    out["dist"] = np.sqrt(out["dist2"])

    out["x0c"] = out["x0"] - out["cx"]
    out["y0c"] = out["y0"] - out["cy"]
    out["z0c"] = out["z0"] - out["cz"]
    out["x1c"] = out["x1"] - out["cx"]
    out["y1c"] = out["y1"] - out["cy"]
    out["z1c"] = out["z1"] - out["cz"]

    out["dxc"] = out["x0c"] - out["x1c"]
    out["dyc"] = out["y0c"] - out["y1c"]
    out["dzc"] = out["z0c"] - out["z1c"]

    eps = 1e-12
    out["r0c2"] = (
        out["x0c"] * out["x0c"] + out["y0c"] * out["y0c"] + out["z0c"] * out["z0c"]
    )
    out["r1c2"] = (
        out["x1c"] * out["x1c"] + out["y1c"] * out["y1c"] + out["z1c"] * out["z1c"]
    )
    out["r0c"] = np.sqrt(out["r0c2"] + eps)
    out["r1c"] = np.sqrt(out["r1c2"] + eps)
    out["r_sum"] = out["r0c"] + out["r1c"]
    out["r_absdiff"] = (out["r0c"] - out["r1c"]).abs()
    out["cos_c0c1"] = (
        out["x0c"] * out["x1c"] + out["y0c"] * out["y1c"] + out["z0c"] * out["z1c"]
    ) / (out["r0c"] * out["r1c"] + eps)

    out["inv_dist"] = 1.0 / (out["dist"] + eps)
    out["inv_dist2"] = 1.0 / (out["dist2"] + eps)
    out["inv_dist3"] = out["inv_dist"] * out["inv_dist2"]

    out["mx"] = 0.5 * (out["x0"] + out["x1"])
    out["my"] = 0.5 * (out["y0"] + out["y1"])
    out["mz"] = 0.5 * (out["z0"] + out["z1"])
    out["mxc"] = out["mx"] - out["cx"]
    out["myc"] = out["my"] - out["cy"]
    out["mzc"] = out["mz"] - out["cz"]
    out["mr2"] = (
        out["mxc"] * out["mxc"] + out["myc"] * out["myc"] + out["mzc"] * out["mzc"]
    )
    out["mr"] = np.sqrt(out["mr2"] + eps)

    out["ux"] = out["dx"] / (out["dist"] + eps)
    out["uy"] = out["dy"] / (out["dist"] + eps)
    out["uz"] = out["dz"] / (out["dist"] + eps)
    out["abs_ux"] = out["ux"].abs()
    out["abs_uy"] = out["uy"].abs()
    out["abs_uz"] = out["uz"].abs()

    az0 = out["atom_0"].astype(str).map(ATOM_Z).astype("float64")
    az1 = out["atom_1"].astype(str).map(ATOM_Z).astype("float64")
    out["az0"] = az0
    out["az1"] = az1
    out["az_sum"] = az0 + az1
    out["az_diff"] = (az0 - az1).abs()
    out["az_prod"] = az0 * az1
    out["az_prod_inv_dist"] = out["az_prod"] * out["inv_dist"]
    out["az_sum_inv_dist2"] = out["az_sum"] * out["inv_dist2"]
    out["az_prod_inv_dist3"] = out["az_prod"] * out["inv_dist3"]

    out["mulliken_sum"] = out["mulliken_0"] + out["mulliken_1"]
    out["mulliken_absdiff"] = (out["mulliken_0"] - out["mulliken_1"]).abs()
    out["mulliken_prod"] = out["mulliken_0"] * out["mulliken_1"]

    out["mulliken_sum_inv_dist"] = out["mulliken_sum"] * out["inv_dist"]
    out["mulliken_absdiff_inv_dist"] = out["mulliken_absdiff"] * out["inv_dist"]
    out["dipole_norm_inv_dist"] = out["dipole_norm"] * out["inv_dist"]
    out["potential_energy_inv_dist"] = out["potential_energy"] * out["inv_dist"]

    return out


train_f = add_pair_features(train)
test_f = add_pair_features(test)

train_f = train_f.merge(
    scc,
    on=["molecule_name", "atom_index_0", "atom_index_1", "type"],
    how="left",
    validate="one_to_one",
    sort=False,
)

miss = train_f[["fc", "sd", "pso", "dso"]].isna().mean()
print("Contribution missing rate in train:", miss.to_dict())

needed_cols = [
    "type",
    "atom_0",
    "atom_1",
    "dx",
    "dy",
    "dz",
    "dist",
    "inv_dist",
    "mr",
    "ux",
    "az0",
    "az1",
    "mulliken_0",
    "mulliken_1",
    "potential_energy",
    "dipole_X",
]
print("Train missing counts (selected):")
print(train_f[needed_cols].isna().sum())
print("Test missing counts (selected):")
print(test_f[needed_cols].isna().sum())



Contribution missing rate in train: {'fc': 0.0, 'sd': 0.0, 'pso': 0.0, 'dso': 0.0}
Train missing counts (selected):
type                0
atom_0              0
atom_1              0
dx                  0
dy                  0
dz                  0
dist                0
inv_dist            0
mr                  0
ux                  0
az0                 0
az1                 0
mulliken_0          0
mulliken_1          0
potential_energy    0
dipole_X            0
dtype: int64
Test missing counts (selected):
type                     0
atom_0              467813
atom_1              467813
dx                  467813
dy                  467813
dz                  467813
dist                467813
inv_dist            467813
mr                  467813
ux                  467813
az0                 467813
az1                 467813
mulliken_0          467813
mulliken_1          467813
potential_energy    467813
dipole_X            467813
dtype: int64


In [6]:
env_base_2 = [
    "degree_2p0",
    "neigh_dist_mean_2p0",
    "neigh_dist_min_2p0",
    "neigh_H_2p0",
    "neigh_C_2p0",
    "neigh_N_2p0",
    "neigh_O_2p0",
    "neigh_other_2p0",
]
env_base_4 = [
    "degree_4p0",
    "neigh_dist_mean_4p0",
    "neigh_dist_min_4p0",
    "neigh_H_4p0",
    "neigh_C_4p0",
    "neigh_N_4p0",
    "neigh_O_4p0",
    "neigh_other_4p0",
]
env_base = env_base_2 + env_base_4

env_0 = env_base
env_1 = [c + "_1" for c in env_base]

for base in env_base:
    a = train_f[base]
    b = train_f[base + "_1"]
    train_f[base + "_sum"] = a + b
    train_f[base + "_absdiff"] = (a - b).abs()

    a = test_f[base]
    b = test_f[base + "_1"]
    test_f[base + "_sum"] = a + b
    test_f[base + "_absdiff"] = (a - b).abs()

env_sym = [b + "_sum" for b in env_base] + [b + "_absdiff" for b in env_base]

ms_base = mag_cols + ["ms_trace", "ms_frob"]
ms_0 = ms_base
ms_1 = [c + "_1" for c in ms_base]

for base in ms_base:
    train_f[base + "_sum"] = train_f[base] + train_f[base + "_1"]
    train_f[base + "_absdiff"] = (train_f[base] - train_f[base + "_1"]).abs()
    train_f[base + "_prod"] = train_f[base] * train_f[base + "_1"]

    test_f[base + "_sum"] = test_f[base] + test_f[base + "_1"]
    test_f[base + "_absdiff"] = (test_f[base] - test_f[base + "_1"]).abs()
    test_f[base + "_prod"] = test_f[base] * test_f[base + "_1"]

ms_sym = (
    [b + "_sum" for b in ms_base]
    + [b + "_absdiff" for b in ms_base]
    + [b + "_prod" for b in ms_base]
)

feature_cols_num = (
    [
        "dx",
        "dy",
        "dz",
        "abs_dx",
        "abs_dy",
        "abs_dz",
        "dist",
        "dist2",
        "dxc",
        "dyc",
        "dzc",
        "inv_dist",
        "inv_dist2",
        "inv_dist3",
        "mr",
        "mr2",
        "ux",
        "uy",
        "uz",
        "abs_ux",
        "abs_uy",
        "abs_uz",
        "az0",
        "az1",
        "az_sum",
        "az_diff",
        "az_prod",
        "az_prod_inv_dist",
        "az_sum_inv_dist2",
        "az_prod_inv_dist3",
        "mulliken_0",
        "mulliken_1",
        "mulliken_sum",
        "mulliken_absdiff",
        "mulliken_prod",
        "mulliken_sum_inv_dist",
        "mulliken_absdiff_inv_dist",
        "dipole_X",
        "dipole_Y",
        "dipole_Z",
        "dipole_norm",
        "dipole_norm_inv_dist",
        "potential_energy",
        "potential_energy_inv_dist",
        "r0c",
        "r1c",
        "r_sum",
        "r_absdiff",
        "cos_c0c1",
    ]
    + ms_0
    + ms_1
    + ms_sym
    + env_0
    + env_1
    + env_sym
)

feature_cols_cat = ["atom_0", "atom_1"]

missing_num = [c for c in feature_cols_num if c not in train_f.columns]
missing_cat = [c for c in feature_cols_cat if c not in train_f.columns]
if missing_num or missing_cat:
    raise KeyError(
        f"Missing feature cols. num missing={missing_num[:10]} cat missing={missing_cat}"
    )

numeric_tf = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ]
)

categorical_tf = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_tf, feature_cols_num),
        ("cat", categorical_tf, feature_cols_cat),
    ],
    remainder="drop",
)

model = Ridge(alpha=3.0)
pipe = Pipeline(steps=[("preprocess", preprocess), ("model", model)])



/tmp/ipykernel_11/3018753419.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_f[base + "_absdiff"] = (test_f[base] - test_f[base + "_1"]).abs()
/tmp/ipykernel_11/3018753419.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_f[base + "_prod"] = test_f[base] * test_f[base + "_1"]
/tmp/ipykernel_11/3018753419.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axi

In [7]:
# Speed: avoid re-running preprocessing 4x per type by fitting/transforming once per type,
# then fitting 4 Ridge models on the same transformed matrices. This preserves the exact
# preprocessing and Ridge solution for each target (same X, same y).
X_tr_all = train_f.loc[:, feature_cols_num + feature_cols_cat].copy()
X_te_all = test_f.loc[:, feature_cols_num + feature_cols_cat].copy()

# Keep categorical as 'category' (no object conversion); OneHotEncoder can handle it.
for c in feature_cols_cat:
    X_tr_all[c] = X_tr_all[c].astype("category")
    X_te_all[c] = X_te_all[c].astype("category")

targets = ["fc", "sd", "pso", "dso"]
preds_comp = {k: np.zeros(len(test_f), dtype=np.float64) for k in targets}

types = sorted(train_f["type"].astype(str).unique().tolist())
print("Training per-type models (contrib targets):", types)

valid_targets_mask = ~train_f[targets].isna().any(axis=1).to_numpy()

train_type_arr = train_f["type"].astype(str).to_numpy()
test_type_arr = test_f["type"].astype(str).to_numpy()

# Reuse the same Ridge hyperparameters; just avoid reconstructing the whole Pipeline each fit.
base_ridge = Ridge(alpha=3.0)

for t in types:
    tr_mask = train_type_arr == t
    te_mask = test_type_arr == t

    if te_mask.sum() == 0:
        continue

    tr_mask_valid = tr_mask & valid_targets_mask
    if tr_mask_valid.sum() == 0:
        continue

    X_tr = X_tr_all.loc[tr_mask_valid, :]
    X_te = X_te_all.loc[te_mask, :]

    # Fit preprocessing once per type
    Xt_tr = preprocess.fit_transform(X_tr)
    Xt_te = preprocess.transform(X_te)

    for k in targets:
        y_tr = train_f.loc[tr_mask_valid, k].to_numpy(dtype=np.float64, copy=False)
        m = Ridge(alpha=base_ridge.alpha)
        m.fit(Xt_tr, y_tr)
        preds_comp[k][te_mask] = m.predict(Xt_te).astype(np.float64, copy=False)

preds = preds_comp["fc"] + preds_comp["sd"] + preds_comp["pso"] + preds_comp["dso"]
print("Pred summary:", pd.Series(preds).describe())



Training per-type models (contrib targets): ['1JHC', '1JHN', '2JHC', '2JHH', '2JHN', '3JHC', '3JHH', '3JHN']
Pred summary: count    467813.000000
mean         13.019680
std          30.800442
min          -8.930362
25%          -1.508953
50%           1.690662
75%           1.690662
max          84.514117
dtype: float64


In [8]:
sub = pd.read_csv(sample_sub_path, usecols=["id"], dtype={"id": np.int32})
pred_df = pd.DataFrame(
    {
        "id": test_f["id"].to_numpy(dtype=np.int32, copy=False),
        "scalar_coupling_constant": preds,
    }
)
sub = sub.merge(pred_df, on="id", how="left", validate="one_to_one")
sub["scalar_coupling_constant"] = (
    sub["scalar_coupling_constant"].fillna(0.0).astype(np.float64)
)

out_path = "submission.csv"
sub.to_csv(out_path, index=False)
print("Wrote:", out_path)
print(sub.head())
print(sub.shape)
print("Submission dtypes:", sub.dtypes.to_dict())

Wrote: submission.csv
        id  scalar_coupling_constant
0  2324604                 84.514117
1  2324605                 -1.508953
2  2324606                  1.690662
3  2324607                  1.690662
4  2324608                 -8.930362
(467813, 2)
Submission dtypes: {'id': dtype('int32'), 'scalar_coupling_constant': dtype('float64')}
